In [ ]:
!pip install opencv-python --break-system-packages

In [ ]:
!pip install fastai --break-system-packages

In [ ]:
import os
from PIL import Image

train_dir = 'Train'

mask_images = []
non_mask_images = []
classes = ['Mask', 'Non Mask']

for class_name in classes:
    class_path = os.path.join(train_dir, class_name)
    
    if os.path.isdir(class_path):
        for count, image_name in enumerate(os.listdir(class_path), start=1):
            image_path = os.path.join(class_path, image_name)
            try:
                image = Image.open(image_path)
                new_name = f"img{count}{os.path.splitext(image_name)[1]}"
                new_path = os.path.join(class_path, new_name)
                os.rename(image_path, new_path)
                
                if class_name == 'Mask':
                    mask_images.append((image, new_name))
                else:
                    non_mask_images.append((image, new_name))
                    
            except Exception as e:
                print(f"Error loading image {image_path}: {e}")

def create_thumbnails(images, size=(256, 256)):
    for idx in range(len(images)):
        img, name = images[idx]
        img.thumbnail(size)
        images[idx] = (img, name)

create_thumbnails(mask_images)
create_thumbnails(non_mask_images)

if mask_images:
    mask_images[0][0].show()
    print(f"Label: Mask, New Name: {mask_images[0][1]}")

if non_mask_images:
    non_mask_images[0][0].show()
    print(f"Label: Non Mask, New Name: {non_mask_images[0][1]}")


In [3]:
from fastai.vision.all import *

In [4]:
for i in range(min(3, len(non_mask_images))):  
    non_mask_images[i][0].show()  
for i in range(min(3, len(non_mask_images))):  
    mask_images[i][0].show()  

In [ ]:
from fastai.vision.all import *

# Define your data and labels
path = Path('Train')
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(256),
    batch_tfms=aug_transforms(size=224)
).dataloaders(path, bs=32)

# Verify the labels
print(dls.vocab)  # This should print ['Mask', 'Non Mask']

# Train the model
learner = vision_learner(dls, resnet34, metrics=accuracy)
learner.fine_tune(2)

In [ ]:
import os
from PIL import Image

test_dir = 'Validation'
test_mask_images = []
test_non_mask_images = []
test_classes = ['Mask', 'Non Mask']

for class_name in test_classes:
    class_path = os.path.join(test_dir, class_name)
    
    if os.path.isdir(class_path):
        for count, image_name in enumerate(os.listdir(class_path), start=1):
            image_path = os.path.join(class_path, image_name)
            try:
                image = Image.open(image_path)
                new_name = f"test_img{count}{os.path.splitext(image_name)[1]}"
                new_path = os.path.join(class_path, new_name)
                os.rename(image_path, new_path)
                
                if class_name == 'Mask':
                    test_mask_images.append((image, new_name))
                else:
                    test_non_mask_images.append((image, new_name))
                    
            except Exception as e:
                print(f"Error loading image {image_path}: {e}")

def create_thumbnails(images, size=(256, 256)):
    for idx in range(len(images)):
        img, name = images[idx]
        img.thumbnail(size)
        images[idx] = (img, name)

create_thumbnails(test_mask_images)
create_thumbnails(test_non_mask_images)

if test_mask_images:
    test_mask_images[0][0].show()
    print(f"Label: Mask, New Name: {test_mask_images[0][1]}")

if test_non_mask_images:
    test_non_mask_images[0][0].show()
    print(f"Label: Non Mask, New Name: {test_non_mask_images[0][1]}")

In [2]:
import random

In [ ]:
i = random.randint(0,(min(len(test_mask_images),len(test_non_mask_images))))
img = test_mask_images[i][0]
is_mask,_,probs = learner.predict(img)
img.show()
print(f"This is a: {is_mask}.")
print(f"Probability it's a Mask: {probs[0]:.4f}")

In [ ]:
i = random.randint(0,(min(len(test_mask_images),len(test_non_mask_images))))
img2 = test_non_mask_images[i][0]
is_mask,_,probs = learner.predict(img2)
img2.show()
print(f"This is a: {is_mask}.")
print(f"Probability it's a Mask: {probs[0]:.4f}")

Real Time FaceMask Detection

In [ ]:
import cv2
import numpy as np
from fastai.vision.all import *
from PIL import Image
import io

learner = learner

cam = cv2.VideoCapture(0)

while True:
    ret, frame = cam.read()
    if not ret:
        break
    
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_frame)
    pil_image = pil_image.resize((224, 224))
    
    is_mask, _, probs = learner.predict(pil_image)
    label = "Mask" if is_mask == 'Mask' else "No Mask"
    confidence = probs[0] if is_mask == 'Mask' else probs[1]
    color = (225, 255, 0) if label == "Mask" else (0, 0, 255)
    
    cv2.putText(frame, f"{label}: {confidence:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.imshow('Face Mask Detection', frame)
    
    if cv2.waitKey(1) == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()